# Домашнє завдання: Прогнозування орендної плати за житло

## Мета завдання
Застосувати знання з лекції для побудови моделі лінійної регресії, що прогнозує орендну плату за житло в Індії. Ви пройдете весь цикл вирішення задачі машинного навчання: від дослідницького аналізу до оцінки якості моделі.

## Опис датасету
**House Rent Prediction Dataset** містить інформацію про 4700+ оголошень про оренду житла в Індії з такими параметрами:
- **BHK**: Кількість спалень, залів, кухонь
- **Rent**: Орендна плата (цільова змінна)
- **Size**: Площа в квадратних футах
- **Floor**: Поверх та загальна кількість поверхів
- **Area Type**: Тип розрахунку площі
- **Area Locality**: Район
- **City**: Місто
- **Furnishing Status**: Стан меблювання
- **Tenant Preferred**: Тип орендаря
- **Bathroom**: Кількість ванних кімнат
- **Point of Contact**: Контактна особа

---

## Завдання 1: Завантаження та перший огляд даних (1 бал)

**Що потрібно зробити:**
1. Завантажте дані з файлу `House_Rent_Dataset.csv`
2. Виведіть розмір датасету
3. Покажіть перші 5 рядків
4. Виведіть загальну інформацію про дані (включно з типами даних та кількістю значень)


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('House_Rent_Dataset.csv', sep=None, engine='python')
df.shape

(4746, 12)

In [2]:
df.head()

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Posted On          4746 non-null   object
 1   BHK                4746 non-null   int64 
 2   Rent               4746 non-null   int64 
 3   Size               4746 non-null   int64 
 4   Floor              4746 non-null   object
 5   Area Type          4746 non-null   object
 6   Area Locality      4746 non-null   object
 7   City               4746 non-null   object
 8   Furnishing Status  4746 non-null   object
 9   Tenant Preferred   4746 non-null   object
 10  Bathroom           4746 non-null   int64 
 11  Point of Contact   4746 non-null   object
dtypes: int64(4), object(8)
memory usage: 445.1+ KB


## Завдання 2: Дослідницький аналіз даних (EDA) (5 балів)

**Що потрібно зробити:**
1. **Аналіз пропущених значень.** Перевірте наявність і відсоток пропущених значень у кожній колонці
2. **Базова статистика.** Обчисліть базову статистику (середнє, квартилі, стандартне відхилення) для числових змінних.
3. **Аналіз цільової змінної.** Побудуйте гістограму розподілу цільової змінної (Rent)
4. **Робота з викидами.** Знайдіть та видаліть викиди в цільовій змінній (якщо є). Визначити викиди можна будь-яким зрозумілим для вас способом, як варіант - таким, що використовується в побудові box-plot (https://en.wikipedia.org/wiki/Box_plot#Example_with_outliers).
5. **Аналіз категоріальних змінних.** Виведіть кількість унікальних значень для кожної з категоріальних колонок.


In [4]:
print('1. Аналіз пропущених значень. Перевірте наявність і відсоток пропущених значень у кожній колонці')
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_percent

1. Аналіз пропущених значень. Перевірте наявність і відсоток пропущених значень у кожній колонці


,0
Posted On,0.0
BHK,0.0
Rent,0.0
Size,0.0
Floor,0.0
Area Type,0.0
Area Locality,0.0
City,0.0
Furnishing Status,0.0
Tenant Preferred,0.0


In [5]:
df.isnull().values.any()

np.False_

In [6]:
print('2. Базова статистика. Обчисліть базову статистику (середнє, квартилі, стандартне відхилення) для числових змінних.')
df.describe().round(2)

2. Базова статистика. Обчисліть базову статистику (середнє, квартилі, стандартне відхилення) для числових змінних.


,BHK,Rent,Size,Bathroom
count,4746.00,4746.00,4746.00,4746.00
mean,2.08,34993.45,967.49,1.97
std,0.83,78106.41,634.20,0.88
min,1.00,1200.00,10.00,1.00
25%,2.00,10000.00,550.00,1.00
50%,2.00,16000.00,850.00,2.00
75%,3.00,33000.00,1200.00,2.00
max,6.00,3500000.00,8000.00,10.00


In [7]:
print('3. Аналіз цільової змінної. Побудуйте гістограму розподілу цільової змінної (Rent)')
fig = px.histogram(
    df,
    x='Rent',
    nbins=500,
    title='Розподіл цільової змінної (Орендна плата)',
    labels={'Rent': 'Орендна плата'}
)
fig.update_layout(
    showlegend=False,
    height=500
)
fig.show()

3. Аналіз цільової змінної. Побудуйте гістограму розподілу цільової змінної (Rent)


In [8]:
print('4. Робота з викидами. Знайдіть та видаліть викиди в цільовій змінній (якщо є).')
Q1 = df['Rent'].quantile(0.25)
Q3 = df['Rent'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"межі викидів: {lower_bound}, {upper_bound}")

outliers = df[(df['Rent'] < lower_bound) | (df['Rent'] > upper_bound)]
print(outliers.shape)

4. Робота з викидами. Знайдіть та видаліть викиди в цільовій змінній (якщо є).
межі викидів: -24500.0, 67500.0
(520, 12)


In [9]:
df_clean = df[(df['Rent'] >= lower_bound) & (df['Rent'] <= upper_bound)]
print("Було:", df.shape)
print("Стало:", df_clean.shape)

Було: (4746, 12)
Стало: (4226, 12)


In [10]:
print('5. Аналіз категоріальних змінних. Виведіть кількість унікальних значень для кожної з категоріальних колонок.')
df_clean.select_dtypes(include='object').nunique()

5. Аналіз категоріальних змінних. Виведіть кількість унікальних значень для кожної з категоріальних колонок.


,0
Posted On,80
Floor,340
Area Type,3
Area Locality,1997
City,6
Furnishing Status,3
Tenant Preferred,3
Point of Contact,3



## Завдання 3: Аналіз кореляцій та взаємозв'язків (3 бали)

**Що потрібно зробити:**
1. Обчисліть матрицю кореляцій для числових змінних
2. Візуалізуйте кореляційну матрицю за допомогою heatmap
3. Побудуйте scatter plot між Size та Rent
4. Проаналізуйте взаємозв'язок між BHK та Rent за допомогою boxplot (який розподіл плати для різних значень BHK)


In [11]:
df_clean.select_dtypes(include='int64').info()

<class 'pandas.core.frame.DataFrame'>
Index: 4226 entries, 0 to 4745
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   BHK       4226 non-null   int64
 1   Rent      4226 non-null   int64
 2   Size      4226 non-null   int64
 3   Bathroom  4226 non-null   int64
dtypes: int64(4)
memory usage: 165.1 KB


In [12]:
print('1. Обчисліть матрицю кореляцій для числових змінних')

metrics_df = df_clean[['BHK', 'Rent', 'Size', 'Bathroom']].dropna()

correlation_matrix = metrics_df.corr()
correlation_matrix


1. Обчисліть матрицю кореляцій для числових змінних


,BHK,Rent,Size,Bathroom
BHK,1.000000,0.401268,0.698453,0.747918
Rent,0.401268,1.000000,0.393605,0.506528
Size,0.698453,0.393605,1.000000,0.680607
Bathroom,0.747918,0.506528,0.680607,1.000000


In [13]:
print('2. Візуалізуйте кореляційну матрицю за допомогою heatmap')
fig = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Кореляція між метриками взаємодії',
    labels=dict(color="Кореляція")
)
fig.update_layout(height=500)
fig.show()

2. Візуалізуйте кореляційну матрицю за допомогою heatmap


In [14]:
print('3. Побудуйте scatter plot між Size та Rent')
fig = px.scatter(
    df_clean,
    x='Size',
    y='Rent',
    title='Залежність Rent від Size',
    labels={'Size': 'Площа', 'Rent': 'Орендна плата'}
)

fig.show()

3. Побудуйте scatter plot між Size та Rent


In [15]:
print('4. Проаналізуйте взаємозвязок між BHK та Rent за допомогою boxplot (який розподіл плати для різних значень BHK)')
fig = px.box(
    df_clean,
    x='BHK',
    y='Rent',
    title='Розподіл орендної плати для різних значень BHK',
    labels={'BHK': 'Кількість кімнат', 'Rent': 'Орендна плата'}
)

fig.show()

4. Проаналізуйте взаємозвязок між BHK та Rent за допомогою boxplot (який розподіл плати для різних значень BHK)


## Завдання 4: Feature Engineering та підготовка даних (4 бали)

**Що потрібно зробити:**
1. Закодуйте категоріальні змінні за допомогою One-Hot Encoding. Пригадайте, що в лекції ми говорили щодо кодування кат. змінних з великої кількістю різних значень і як працювати з такими випадками. Ви можете закодувати не всі кат. змінні, а лише ті, що вважаєте за потрібні (скажімо ті, що мають відносно небагато різних значень).
2. **Опціонально (по 0.5 бала за кожну доцільну ознаку):** Додайте нові ознаки, обчислені на основі наявних даних, які б на ваш погляд були корисними для моделі
3. Виберіть ознаки для побудови моделі (виключіть непотрібні колонки). Виключити можна, наприклад, ті колонки, які мають категоріальний тип і забагато (більше 20) різних значень. Треба виключити хоча б 1 колонку.
4. Розділіть дані на ознаки (X) та цільову змінну (y)
5. Застосуйте стандартизацію до числових ознак


In [16]:
df_clean.select_dtypes(include='object').nunique()

,0
Posted On,80
Floor,340
Area Type,3
Area Locality,1997
City,6
Furnishing Status,3
Tenant Preferred,3
Point of Contact,3


In [17]:
print(df_clean['Area Type'].unique())
print(df_clean['City'].unique())
print(df_clean['Furnishing Status'].unique())
print(df_clean['Tenant Preferred'].unique())


['Super Area' 'Carpet Area' 'Built Area']
['Kolkata' 'Mumbai' 'Bangalore' 'Delhi' 'Chennai' 'Hyderabad']
['Unfurnished' 'Semi-Furnished' 'Furnished']
['Bachelors/Family' 'Bachelors' 'Family']


In [18]:
print('1. Закодуйте категоріальні змінні за допомогою One-Hot Encoding.')
print('Пригадайте, що в лекції ми говорили щодо кодування кат. змінних з великої кількістю різних значень і як працювати з такими випадками.')
print('Ви можете закодувати не всі кат. змінні, а лише ті, що вважаєте за потрібні (скажімо ті, що мають відносно небагато різних значень).')

area_dummies = pd.get_dummies(df_clean['Area Type'], prefix='Area')
city_dummies = pd.get_dummies(df_clean['City'], prefix='City')
status_dummies = pd.get_dummies(df_clean['Furnishing Status'], prefix='Status')
tenant_dummies = pd.get_dummies(df_clean['Tenant Preferred'], prefix='Tenant')

df_clean = df_clean.drop(columns=['Area Type', 'City', 'Furnishing Status', 'Tenant Preferred'])

df_clean = pd.concat([df_clean, area_dummies,city_dummies,status_dummies,tenant_dummies], axis=1)
df_clean.columns

1. Закодуйте категоріальні змінні за допомогою One-Hot Encoding.
Пригадайте, що в лекції ми говорили щодо кодування кат. змінних з великої кількістю різних значень і як працювати з такими випадками.
Ви можете закодувати не всі кат. змінні, а лише ті, що вважаєте за потрібні (скажімо ті, що мають відносно небагато різних значень).


Index(['Posted On', 'BHK', 'Rent', 'Size', 'Floor', 'Area Locality',
       'Bathroom', 'Point of Contact', 'Area_Built Area', 'Area_Carpet Area',
       'Area_Super Area', 'City_Bangalore', 'City_Chennai', 'City_Delhi',
       'City_Hyderabad', 'City_Kolkata', 'City_Mumbai', 'Status_Furnished',
       'Status_Semi-Furnished', 'Status_Unfurnished', 'Tenant_Bachelors',
       'Tenant_Bachelors/Family', 'Tenant_Family'],
      dtype='object')

In [19]:
print('2. Опціонально (по 0.5 бала за кожну доцільну ознаку): Додайте нові ознаки, обчислені на основі наявних даних, які б на ваш погляд були корисними для моделі')

print('1). Ціна за квадратний фут')
df_clean['Rent_per_Size'] = df_clean['Rent'] / df_clean['Size']
print('2). Вартість однієї кімнати')
df_clean['Rent_per_BHK'] = df_clean['Rent'] / df_clean['BHK']
print('3). Скільки кімнат на одиницю площі (тісно чи просторо)')
df_clean['BHK_per_Size'] = df_clean['BHK'] / df_clean['Size']
print('4). Скільки ванних кімнат на одну спальню')
df_clean['Bathroom_per_BHK'] = df_clean['Bathroom'] / df_clean['BHK']
print('5). Скільки спалень на одну ванну кімнату')
df_clean['BHK_per_Bathroom'] = df_clean['BHK'] / df_clean['Bathroom']
print('6). Чи багато ванних кімнат')
df_clean['Many_Bathrooms'] = (df_clean['Bathroom'] >= 2).astype(int)
print('7). Скільки кімнат всього в квартирі')
df_clean['All_rooms'] = (df_clean['Bathroom'] + df_clean['BHK'])
print('8). Чи дорожча квартира за середню вартість')
df_clean['High_Rent'] = (df_clean['Rent'] > df_clean['Rent'].median())

2. Опціонально (по 0.5 бала за кожну доцільну ознаку): Додайте нові ознаки, обчислені на основі наявних даних, які б на ваш погляд були корисними для моделі
1). Ціна за квадратний фут
2). Вартість однієї кімнати
3). Скільки кімнат на одиницю площі (тісно чи просторо)
4). Скільки ванних кімнат на одну спальню
5). Скільки спалень на одну ванну кімнату
6). Чи багато ванних кімнат
7). Скільки кімнат всього в квартирі
8). Чи дорожча квартира за середню вартість


In [20]:
#df_clean.select_dtypes(include='int64').info()

In [21]:
#df_clean[(df_clean['Bathroom'] - df_clean['BHK']) >= 2][['BHK', 'Bathroom', 'Size', 'Rent']].sort_values(by='Bathroom', ascending=False)

In [22]:
print('3. Виберіть ознаки для побудови моделі (виключіть непотрібні колонки). ')
print('Виключити можна, наприклад, ті колонки, які мають категоріальний тип і забагато (більше 20) різних значень. Треба виключити хоча б 1 колонку.')

print(df_clean.select_dtypes(include='object').nunique())
df_model = df_clean.drop(columns=['Area Locality','Posted On', 'Floor', 'Point of Contact'])
df_model.select_dtypes(include='object').nunique()


3. Виберіть ознаки для побудови моделі (виключіть непотрібні колонки). 
Виключити можна, наприклад, ті колонки, які мають категоріальний тип і забагато (більше 20) різних значень. Треба виключити хоча б 1 колонку.
Posted On             80
Floor                340
Area Locality       1997
Point of Contact       3
dtype: int64


,0


In [23]:
print('4. Розділіть дані на ознаки (X) та цільову змінну (y)')

X = df_model.drop(columns=['Rent'])
y = df_model['Rent']

print(f"\nРозмір X (ознак): {X.shape}")
print(f"Розмір y (цілі): {y.shape}")

4. Розділіть дані на ознаки (X) та цільову змінну (y)

Розмір X (ознак): (4226, 26)
Розмір y (цілі): (4226,)


In [24]:
print('5. Застосуйте стандартизацію до числових ознак')

from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
X_scaled_df.head()

5. Застосуйте стандартизацію до числових ознак


,BHK,Size,Bathroom,Area_Built Area,Area_Carpet Area,Area_Super Area,City_Bangalore,City_Chennai,City_Delhi,City_Hyderabad,...,Tenant_Bachelors/Family,Tenant_Family,Rent_per_Size,Rent_per_BHK,BHK_per_Size,Bathroom_per_BHK,BHK_per_Bathroom,Many_Bathrooms,All_rooms,High_Rent
0,0.052966,0.469859,0.272578,-0.02176,-0.878583,0.879429,-0.502144,-0.501774,-0.387614,-0.499926,...,0.597576,-0.315692,-0.469635,-0.690935,-0.230881,0.147328,-0.358513,0.729187,0.171277,-0.902935
1,0.052966,-0.147778,-1.133910,-0.02176,-0.878583,0.879429,-0.502144,-0.501774,-0.387614,-0.499926,...,0.597576,-0.315692,-0.126157,-0.047545,-0.128728,-1.637671,2.242635,-1.371389,-0.562818,1.107499
2,0.052966,0.263980,-1.133910,-0.02176,-0.878583,0.879429,-0.502144,-0.501774,-0.387614,-0.499926,...,0.597576,-0.315692,-0.298878,-0.240562,-0.203640,-1.637671,2.242635,-1.371389,-0.562818,1.107499
3,0.052966,-0.147778,-1.133910,-0.02176,-0.878583,0.879429,-0.502144,-0.501774,-0.387614,-0.499926,...,0.597576,-0.315692,-0.396033,-0.690935,-0.128728,-1.637671,2.242635,-1.371389,-0.562818,-0.902935
4,0.052966,-0.044839,-1.133910,-0.02176,1.138197,-1.137102,-0.502144,-0.501774,-0.387614,-0.499926,...,-1.673428,-0.315692,-0.475408,-0.851782,-0.150761,-1.637671,2.242635,-1.371389,-0.562818,-0.902935


In [25]:
X_scaled_df.columns

Index(['BHK', 'Size', 'Bathroom', 'Area_Built Area', 'Area_Carpet Area',
       'Area_Super Area', 'City_Bangalore', 'City_Chennai', 'City_Delhi',
       'City_Hyderabad', 'City_Kolkata', 'City_Mumbai', 'Status_Furnished',
       'Status_Semi-Furnished', 'Status_Unfurnished', 'Tenant_Bachelors',
       'Tenant_Bachelors/Family', 'Tenant_Family', 'Rent_per_Size',
       'Rent_per_BHK', 'BHK_per_Size', 'Bathroom_per_BHK', 'BHK_per_Bathroom',
       'Many_Bathrooms', 'All_rooms', 'High_Rent'],
      dtype='object')

## Завдання 5: Розділення даних та навчання моделі (3 бали)

**Що потрібно зробити:**
1. Розділіть дані на навчальну (80%) та тестову (20%) вибірки.
2. Створіть модель лінійної регресії.
3. Навчіть модель на навчальних даних.
4. Виведіть усі коефіцієнти моделі (ваги) та напишіть, які 2 ознаки найбільше впливають на прогноз.
5. Зробіть прогнози на тренувальній та тестовій вибірках.

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y,
    test_size=0.2,  # 20% даних йде на тест
    random_state=42  # фіксуємо випадковість для відтворюваності
)

In [27]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()

In [28]:
model.fit(X_train, y_train)

LinearRegression()

In [29]:
for feature, weight in zip(model.feature_names_in_, model.coef_):
    print(f"{feature}: {weight:.2f}")

print(f"\nЗміщення (intercept): {model.intercept_:.2f}")

BHK: -605.71
Size: 1558.16
Bathroom: 4483.23
Area_Built Area: 76.25
Area_Carpet Area: 194.37
Area_Super Area: -197.69
City_Bangalore: 114.48
City_Chennai: -271.28
City_Delhi: 115.45
City_Hyderabad: -454.81
City_Kolkata: -120.74
City_Mumbai: 701.22
Status_Furnished: -103.80
Status_Semi-Furnished: -58.13
Status_Unfurnished: 128.44
Tenant_Bachelors: 88.58
Tenant_Bachelors/Family: -50.99
Tenant_Family: -38.37
Rent_per_Size: 1472.59
Rent_per_BHK: 10044.24
BHK_per_Size: -903.64
Bathroom_per_BHK: -4283.51
BHK_per_Bathroom: -285.43
Many_Bathrooms: 298.76
All_rooms: 2008.21
High_Rent: 913.60

Зміщення (intercept): 19296.00


**Чим більше значення, тим сильніший вплив. Це Rent_per_BHK(10044.24) і Bathroom(4483.23)**

In [30]:
# Прогнози на навчальній вибірці
y_train_pred = model.predict(X_train)

# Прогнози на тестовій вибірці (нові дані!)
y_test_pred = model.predict(X_test)

# Порівняння перших 10 прогнозів з реальністю
comparison = pd.DataFrame({
    'Реальна оренда': y_test.values[:10],
    'Прогнозована оренда': y_test_pred[:10].round(0),
    'Помилка': (y_test.values[:10] - y_test_pred[:10]).round(0)
})
print("Приклади прогнозів на тестовій вибірці:")
print(comparison)


Приклади прогнозів на тестовій вибірці:
   Реальна оренда  Прогнозована оренда  Помилка
0           22000              25042.0  -3042.0
1            5000                739.0   4261.0
2           37000              36010.0    990.0
3            8000               4674.0   3326.0
4           15000              14727.0    273.0
5           20000              22680.0  -2680.0
6            8500               8242.0    258.0
7            7000               3989.0   3011.0
8            3000              -2285.0   5285.0
9            8000               7406.0    594.0


Бачимо, що модель не є надто точною, більшість прогнозів близькі, інколи великі помилки

---



## Завдання 6: Оцінка якості моделі (2 бали)

**Що потрібно зробити:**
1. Обчисліть MAE, RMSE та R² для навчальної та тестової вибірок
2. Порівняйте метрики та зробіть висновок про якість моделі
3. Проаналізуйте і дайте висновок, чи є ознаки перенавчання або недонавчання (**Нагадування**: перенавчання - коли модель дуже добре працює на тренувальних даних, але погано на тестових; недонавчання - коли модель погано працює навіть на тренувальних даних)
4. Побудуйте графік розсіювання "реальні vs прогнозовані значення" та зробіть висновок про якість моделі


In [31]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Розраховуємо метрики для тестової вибірки
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_test_pred)

print("="*50)
print("МЕТРИКИ ЯКОСТІ МОДЕЛІ (на тестовій вибірці):")
print("="*50)
print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

# Порівняння з навчальною вибіркою
mae = mean_absolute_error(y_train, y_train_pred)
mse = mean_squared_error(y_train, y_train_pred)
rmse = np.sqrt(mse)
r2_train = r2_score(y_train, y_train_pred)

print("="*50)
print("МЕТРИКИ ЯКОСТІ МОДЕЛІ на тренувальній вибірці:")
print("="*50)
print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f} ")
print(f"R²: {r2_train:.3f}")
print("="*50)
print(np.mean(y_train))

МЕТРИКИ ЯКОСТІ МОДЕЛІ (на тестовій вибірці):

MAE: 3072.83
RMSE: 4425.84
R²: 0.898
МЕТРИКИ ЯКОСТІ МОДЕЛІ на тренувальній вибірці:

MAE: 3080.05
RMSE: 4472.35 
R²: 0.895
19349.54852071006


**У нас модель працює стабільно на train і test. Немає ознак перенавчання і немає ознак слабкої моделі**

In [32]:
# Візуалізація: реальні vs прогнозовані значення
fig = px.scatter(
    x=y_test,
    y=y_test_pred,
    title='Реальна vs Прогнозована оренда (тестова вибірка)',
    labels={'x': 'Реальна оренда', 'y': 'Прогнозована оренда'},
    opacity=0.6
)

# Додаємо ідеальну лінію (де прогноз = реальність)
max_val = max(y_test.max(), y_test_pred.max())
fig.add_trace(
    go.Scatter(
        x=[0, max_val],
        y=[0, max_val],
        mode='lines',
        name='Ідеальний прогноз',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(height=500)
fig.show()

## Завдання 7: Аналіз помилок (4 бали)

**Що потрібно зробити:**
1. Обчисліть помилки (residuals = реальні - прогнозовані значення)
2. Побудуйте гістограму розподілу помилок
3. Створіть scatter plot помилок відносно величини прогнозованих значень. Чи росте помилка з ростом прогнозованого значення?
4. Знайдіть 5 прогнозів з найбільшими помилками
5. Проаналізуйте, на яких типах житла модель помиляється найбільше. Типи можна розрізняти за кількістю кімнат чи містом, наприклад.
6. Подумайте і напишіть, які наступні кроки ви б зробили, аби поліпшити якість моделі. Опціонально можна їх зробити і ми перевіримо :)

In [33]:
# Розраховуємо помилки (залишки)
residuals = y_test - y_test_pred

# Гістограма помилок
fig = px.histogram(
    x=residuals,
    nbins=50,
    title='Розподіл помилок прогнозування',
    labels={'x': 'Помилка (реальні - прогнозовані)', 'count': 'Кількість'},
    color_discrete_sequence=['#e74c3c']
)
fig.add_vline(x=0, line_dash="dash", line_color="black", annotation_text="Ідеальний прогноз")
fig.update_layout(height=400)
fig.show()

In [34]:
# Scatter plot: помилки vs прогнозовані значення
fig = px.scatter(
    x=y_test_pred,
    y=residuals,
    title='Залежність помилок від прогнозованих значень',
    labels={'x': 'Прогнозовані лайки', 'y': 'Помилка'},
    opacity=0.5
)

# Додаємо горизонтальну лінію на 0
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Без помилки")

fig.update_layout(height=400)
fig.show()

In [35]:
# Знаходимо оголошення з найбільшими помилками
errors_df = pd.DataFrame({
    'real': y_test.values,
    'predicted': y_test_pred,
    'error': np.abs(residuals)
})

# Топ-5 найбільших помилок
top_errors = errors_df.nlargest(5, 'error')
print("Оголошення з найбільшими помилками прогнозування:")
print(top_errors)

Оголошення з найбільшими помилками прогнозування:
       real     predicted         error
809   60000  81295.420057  21295.420057
954   59000  79831.500971  20831.500971
1371  55000  73915.155704  18915.155704
3520  65000  46192.977365  18807.022635
1066  52000  70320.336276  18320.336276


In [36]:
df.loc[[809, 954, 1371, 3520, 1066]]

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
809,2022-06-30,1,60000,455,18 out of 22,Carpet Area,"Sugee Sadan, Dadar West",Mumbai,Unfurnished,Bachelors/Family,1,Contact Agent
954,2022-06-30,1,59000,470,26 out of 42,Carpet Area,"Lodha New Cuffe Parade, Bhakti Park",Mumbai,Semi-Furnished,Bachelors,1,Contact Agent
1371,2022-06-30,1,55000,505,4 out of 12,Carpet Area,"Godrej The Trees, Vikhroli East",Mumbai,Semi-Furnished,Bachelors/Family,1,Contact Agent
3520,2022-07-10,3,65000,1444,11 out of 14,Super Area,Nungambakkam,Chennai,Semi-Furnished,Bachelors,3,Contact Agent
1066,2022-05-23,1,52000,475,10 out of 18,Carpet Area,Shivaji Park,Mumbai,Unfurnished,Bachelors/Family,1,Contact Agent
